# သင်ခန်းစာ ၁၈: ဝိဇ္ဇာဆိုင်ရာ ဦးဆောင်သူများကို သိပ်သည်းသော မှတ်ချက်များဖြင့် လုံခြုံရေး

## လက်တွေ့လုပ်ဆောင်ရေး အမှတ်တရများ

ဤ စာရွက်တွင် လုပ်ဆောင်ရန် အိမ်ဆုံးလုပ်ငန်းလေးခု ပါဝင်သည်-

1. **ဦးဆောင်သူကိရိယာ ခေါ်ဆိုမှု အတွက် သင်၏ ပထမဆုံး မှတ်ချက်အား လက်မှတ်ရေးထိုးပါ** နှင့် ထိုသည်ကို စစ်ဆေးပါ။
2. **မှတ်ချက်ကို လိမ်ညာမှု ပြုလုပ်ပြီး စစ်ဆေးမှု မအောင်မြင်မှုကို ကြည့်ရှုပါ။
3. **မှတ်ချက်သုံးခု ပါဝင်သည့် ချိန်းကြိုးတစ်ခု တည်ဆောက်ပြီး ချိန်းကြိုးမှန်ကန်မှုကို အတည်ပြုပါ။
4. **Microsoft Agent Framework ကိရိယာ ခေါ်ဆိုမှုကို လွှမ်းမိုးပြီး လုပ်ဆောင်ချက်တိုင်းမှ ပြန်လည်ထုတ်ပေးသော မှတ်ချက်ကို ထုတ်ဆောင်ပါ။

အားလုံးသော သိပ်သည်း သည်း မှတ်ချက်ပဒေသာ ပုံစံများကို (Ed25519 အတွက် `pynacl`, RFC 8785 canonical JSON အတွက် `jcs`, SHA-256 အတွက် Python စံသတ်မှတ်စာကြည့်တိုက်မှ `hashlib`) ကောင်းမွန်စီမံထားသည့် စာကြည့်တိုက်များမှ တင်သွင်းထားသည်။ မှတ်ချက် ရုပ်ပိုင်းဟာ ရိုးရှင်းတဲ့ Python ဖြစ်ပြီး သင် ဖတ်၍ ပြင်ဆင်နိုင်သည်။

စားလို့ စာလုံးစီလက်လည်းဆက်တိုက် တစ်ခုချင်းတည်း ဖတ်ပါ။ ထိုအပိုင်းတိုင်းသည် အတိုချုပ်ပြီး တစ်ဦးကိုယ်တိုင် ဆောင်ရွက်နိုင်သည်။


## Setup

အောက်ပါ dependencies နှစ်ခုကို install လုပ်ပါ။ နှစ်ခုလုံးသည် permissive license များ (Apache-2.0 / MIT) ရှိသည်။


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

ဤအကူအညီနှစ်ခုသည် base64url encoding (padding မပါဘဲ) နှင့် arbitrary objects များအတွက် SHA-256 hashing ကို ကိုင်တွယ်ပေးသည်။ ၎င်းတို့က notebook အခြားအစိတ်အပိုင်းများကို receipt logic အတွက်သာ အာရုံစိုက်စေသည်။


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: သင့်ပထမသောလက်မှတ်ထိုးထားသော လက်ခံလွှာကို မှတ်တမ်းတင်ပါ

**Contoso Travel** အတွက် အေးဂျင့်သည် စစ်စစ် Sydney မှ Los Angeles သို့ ခရီးစဉ်ကို ရှာဖွေခဲ့သည်ဟု ထင်ပါစို့။ ဤကိရိယာခေါ်ဆိုမှုကို လက်မှတ်ထိုးထားသော လက်ခံလွှာအဖြစ် မှတ်တမ်းတင်လိုသည်၊ ဒါမှ အနာဂတ်လူစစ်ဆေးသူတစ်ဦးက ကျွန်တော်တို့ကို မယုံကြည့်ပဲ တရားဝင်စစ်ဆေးနိုင်ပါလိမ့်မည်။

### Step 1.1: လက်မှတ်ထိုးရန်သော key ကို ဖန်တီးပါ

ထုတ်လုပ်ရေးတွင် အေးဂျင့်၏ လက်မှတ်ထိုးရန် key သည် hardware security module (HSM), Azure Key Vault သို့မဟုတ် ဒါ့လို ကာကွယ်ထားသော ထိန်းသိမ်းရာတစ်ခုတွင် လျှို့ဝှက်ထားပါမည်။ သင်ခန်းစာဤတွင် သင့်ကြိုက်သော key အသစ်တစ်ခုကို memory ထဲတွင် ဖန်တီးသည်။


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### အဆင့် 1.2: လက်ခံလွှာအစိတ်အပိုင်းကို ဆောက်လုပ်ခြင်း

payload တွင် လက်ခံလွှာသည် အတည်ပြုရန်လိုသော အရာအားလုံးပါဝင်သည်။ မည်သူက လုပ်ဆောင်ခဲ့သည်၊ ဘယ်ကိရိယာ၊ ဘယ်အမှုများနှင့်၊ ဘာတွေပြန်လာခဲ့သည်၊ မည်သည့်မူဝါဒအောက်မှ၊ ဘယ်အချိန်တွင်ဆိုသည်တို့ပါဝင်သည်။ arguments နှင့် ရလဒ်ကို inline ထဲတွင် ထည့်သွင်းခြင်းမပြုဘဲ ထိုအရာများကို hash ပြုလုပ်ပြီး လက်ခံလွှာတွင် အချက်အလက်များ ထွက်သွားခြင်း မဖြစ်စေရန် ဆောင်ရွက်သည်။


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### အဆင့် 1.3: လက်မှတ်ရေးထိုး၍ လက်ခံမှတ်တမ်းကိုစုစည်းခြင်း

အဆင့်သုံးအဆင့်ရှိသည် -

1. JCS ကို အသုံးပြု၍ payload ကို canonicalize လုပ်ပါ၊ ဒါကြောင့် အနောက်တူညီသော receipt ကို ထုတ်ပေးသော အပြုအမူနှစ်ခုသည် ဘိုင်းမြို့တူညီသည့် bytes များကို ထုတ်ပေးသည်။
2. SHA-256 ဖြင့် canonical bytes ကို hash လုပ်ပါ။
3. Ed25519 ကိုယ်ပိုင် key ဖြင့် hash ကို လက်မှတ်ရေးထိုးပါ။

ထိုလက်မှတ်ကို မူရင်း payload နှင့် ချိတ်ဆက်ပြီး နောက်ဆုံး receipt ကို ထုတ်ပေးသည်။


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### အဆင့် ၁.၄: ချွန်ချန်းလက္ခံပြုမႈကို စစ်ဆေးပါ

စစ်ဆေးမှုသည် လုပ်ဆောင်ချက်ကို နောက်ပြန် လှည့်ပြန်ခြင်း ဖြစ်သည်။ ကျွန်ုပ်တို့သည် လက်မှတ်ကို ဖယ်ရှားပြီး၊ လွန်စီးဖွဲ့စည်းတည်ဆောက်ထားသော hash ကို ထပ်မံတွက်ချက်ပြီး၊ လက်မှတ်ကို လက်ခံရရှိသောအရေးအသားတွင် ပါဝင်သည့် စမြစ်သောသော သော့နှင့် နှိုင်းယှဉ် စစ်ဆေးပါသည်။

ဤစစ်ဆေးမှုကို လုပ်ဆောင်နေသော စစ်ဆေးသူသည် ကျွန်ုပ်တို့ထံမှ လက်ခံရရှိမှု အချက်အလက်မှသာ လိုအပ်သည်။ မည်သည့် ဝန်ဆောင်မှုကို ခေါ်ယူရန်မပြုရ၊ သော့ချက် သိမ်းဆည်းရာ ဒေသကို မေးမြန်းရန်မလို၊ ယုံကြည်မှုတစ်ခုလုံးမလို။


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

`Receipt is valid: True` ကိုတွေ့ရမယ်။ အေးဂျင့်က သူ့ရဲ့ ပထမဆုံး စာချုပ်ရေးနည်းဖြင့် လက်မှတ်ရေးထိုးထားတဲ့ စစ်ဆေးမှတ်တမ်းကို ထုတ်လုပ်လိုက်ပြီဖြစ်တယ်။


## Section 2: လက်မှတ်နှင့် ချုပ်ဆွဲခြင်း

လက်မှတ်များ၏ အဓိကရည်ရွယ်ချက်မှာ ၎င်းများသည် ချုပ်ဆွဲထားသည်ဟု သက်သေပြနိုင်စေရန်ဖြစ်သည်။ ဤကို သက်သေပြကြ စို့။

ကျွန်ုပ်တို့သည် လက်မှတ်တစ်ခု၏ အက္ခရာတစ်လုံးကို တိတိကျကျ ပြောင်းလဲပြီး သက်သေခံခြင်း မအောင်မြင်ကြောင်း ကြည့်မည်။


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### မဘာဖြစ်သွားလဲ?

`policy_id` ကိုပြောင်းလိုက်တဲ့အခါ canonical bytes တွေပြောင်းခဲ့တယ်။ အဲ့ဒီ bytes တွေရဲ့ SHA-256 hash က ပြောင်းသွားတယ်။ signature (မူလ hash ပေါ်မှာရှိတာ) ကတော့ hash အသစ်နဲ့ မကိုက်ညီတော့ဘူး။ Verification ကတော့ တိတိကျကျ `False` ကို return ပြန်ပြတယ်။

receipt ရဲ့ field တစ်ခုခုကို ပြင်လိုက်လို့လည်း verify လုပ်နိုင်ဖို့မရှိပါဘူး၊ attacker က private key ကိုမိမိကိုယ်ပိုင်ထိန်းထားတာမဟုတ်ရင်ပဲ။ private key က key vault ထဲမှာဖြစ်နေတာနဲ့ public key ကို ထုတ်ပြန်ထားတာ မြှန်မှ ရိုက်ခတ်တာတွေ ဖုံးကွယ်နိုင်ခြင်း မရှိပါဘူး။

ကိုယ့်ကိုယ်တိုင်ကြိုးစားပါ: အပေါ်က cell ထဲမှာ `tool_name` သို့မဟုတ် `agent_id` သို့မဟုတ် `timestamp` ကိုပြင်ပြီး နောက်တခေါက် ပြန် run ကြည့်ပါ။ ဘယ် ပြောင်းလဲမှုမျိုးမှ မမှန်ကန်တဲ့ receipt ကိုပဲ ဖြစ်စေမှာပါ။


## အပိုင်း ၃။ ချိတ်ဆက်ခံရလက်မှတ်များကို တစ်ပြိုင်နက်ချိတ်ဆက်ခြင်း

တစ်ခုလက်မှတ်နှင့် တစ်ခုလုပ်ဆောင်ချက်ကိုသာ ကာကွယ်ပေးသည်။ အုပ်စုအများအပြားလုပ်ဆောင်ချက်များ လုပ်ဆောင်သည့် အေဂျင့်အများအပြား ရှိသည်။ အစဉ်အလာတစ်ခုလုံးကို ကပ်တိုင်တားတားထားရေးအတွက်၊ အသစ်သောလက်မှတ်၏ ပေးသွင်းချက်ထဲတွင် ယခင်လက်မှတ်၏ ဟက်ရှ်ကို ထည့်သွင်းခြင်းဖြင့် ချိတ်ဆက်သည်။

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```
  
သူမည်သူမဆို လက်မှတ်တစ်ခုကို ဖယ်ရှားလိုက်ခြင်း သို့မဟုတ် အဆက်လက်အတိုင်း မလိုက်လျောညီထွေစွာ စီစဉ်မထားပါက ချိတ်ဆက်မှုသည် အတိအကျ ထိုနေရာတွင် ဆက်တိုက်မှု ဖျက်ဆီးပျက်ယွင်းသွားမည်ဖြစ်သည်။ နောက်ပိုင်းအနက် အချို့သောလက်မှတ်များ၏ အတည်ပြုမှု အောင်မြင်မှု မရရှိတော့ပါဘူး၊ အကြောင်းမှာ ၎င်းတို့ `previous_receipt_hash` သည် ၎င်း၏ ယခင်လက်မှတ်၏ စစ်မှန်သော ဟက်ရှ်နှင့်မကိုက်ညီတော့ခြင်းကြောင့် ဖြစ်သည်။


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ယခု ချိတ်ဆက်မှုကို ပျက်စီးအောင် အလယ်ပိုင်း တင်ပြချက်ကို လိမ်လည် ပြုပြင်ပြီး ပြန်လည်စစ်ဆေးပါ။ လိမ်လည် ပြုပြင်ထားသော တင်ပြချက်သည် လက်မှတ်စစ်ဆေးမှုမှာ ကျရှုံးပြီး၊ နောက်တင်ပြချက်သည် ချိတ်ဆက်မှုစစ်ဆေးမှုမှာလည်း ကျရှုံးသည် (အကြောင်းက `previous_receipt_hash` သည် ပြုပြင်ထားသော အလယ်ပိုင်း တင်ပြချက်၏ ဟက်ရှ်နှင့် မကိုက်ညီတော့လို့ ဖြစ်သည်)။


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 သည် မပြောင်းလဲထားပါက (ပြောင်းလဲမှုမရှိသည့် အခြေခံစာရင်း မရှိပါ) စစ်ဆေးမှုကို အောင်မြင်စွာ ပြုလုပျသည်။ Receipt 1 သည် `tool_args_hash` ကို ပြောင်းလဲထားသောကြောင့် အထောက်အထား စစ်ဆေးမှုမအောင်မြင်ပါ။ Receipt 2 သည် `previous_receipt_hash` ကို မူလ (ယခု ပြောင်းလဲထားသော) receipt 1 အပေါ်မှတွက်ချက်ထားသောကြောင့် ချိတ်ဆက်မှု စစ်ဆေးမှု မအောင်မြင်ပါ။

တိုက်ခိုက်သူမှ ပြောင်းလဲထားသော receipt 1 ကို ထပ်မံ လက်မှတ်ထိုးနိုင်ခဲ့ပါကပါ (မည်သည့်အခါမှ ပုဂ္ဂလိက key မရှိဘဲ မလုပ်နိုင်ပါ) Receipt 2 ရဲ့ ချိတ်ဆက်မှု မကိုက်ညီမှုကြောင့် ပြောင်းလဲမှုကို ဖော်ထုတ်နိုင်ပါလိမ့်မည်။ ပြောင်းလဲမှုကို မှောင်မိုက်ရန်အတွက် လူသည် ပြောင်းလဲမှုပြုခဲ့သော အချက်မှစ၍ receipt တစ်စုံတစ်ရာအားလုံးကို ထပ်မံလက်မှတ်ထိုးရမည်ဖြစ်ပြီး ၎င်းသည် ပုဂ္ဂလိက key ကို ပိုင်ဆိုင်မှုလိုအပ်ပါသည်။


## အပိုင်း ၄: အေးဂျင့်တူးလ်ခေါ်ဆိုမှုကို လက်မှတ်စာရင်းသွင်းခြင်းဖြင့် ဝတ်ဆင်ခြင်း

အမှန်တကယ် တပ်ဆင်ရာတွင်၊ အေးဂျင့်စာရေးသူတိုင်းသည် `make_receipt` ကို ခေါ်ရန် မေ့နေချင်ကြဘူး။ အထူးသဖြင့် တူးလ်တိုင်းခေါ်သုံးရာတွင် လက်မှတ်စာရင်းသွင်းခြင်းကို အလိုအလျောက်ဖြစ်စေရန်သာလိုသည်။

ဒီမှာ အလွယ်ဆုံးပုံစံတစ်ခုရှိသည်- မည်သည့် callable tool function ကိုမဆို လက်ခံ၍ လက်မှတ်ထုတ်ပေးသည့် ဗားရှင်းကို ပြန်လည်ထုတ်ပေးသော wrapper class တစ်ခု ဖြစ်သည်။ ဤသည်သည် Microsoft Agent Framework (`agent_framework.azure`) အပါအဝင် မည်သည့် agent framework ကိုမဆို ကိုက်ညီနိုင်သည်။

သင်တွင် Azure AI Foundry စီမံကိန်းမရှိပါက၊ အောက်ပါ ဒေသတွင်း mock သည် အဆိုပါပုံစံကို ဖော်ပြပေးသည်။


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework နှင့် ပေါင်းစပ်ခြင်း

အထက်ပါ `ReceiptedTool` wrapper သည် framework-agnostic ဖြစ်ပါသည်။ Microsoft Agent Framework ဖြင့် တည်ဆောက်ထားသော agent အတွင်း၌ အသုံးပြုရန်အတွက် ဖမ်းဆီးထားသော function ကို tool အဖြစ် မှတ်ပုံတင်ပါ။ အဆိုပါ sketch (သင်သည် mock ကို အမှန်တကယ် Azure AI Foundry tool မှတ်ပုံတင်မှုဖြင့် အစားထိုးမည်)။

```python
# ပုံစံပေါင်းစည်းမှုကိုပြသနေသော မျှမော့ကုတ်။
# agent_framework.azure မှ AzureAIProjectAgentProvider ကိုတင်သွင်းခြင်း
# azure.identity မှ AzureCliCredential ကိုတင်သွင်းခြင်း
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="သင်သည် Contoso Travel အေးဂျင့့်ဖြစ်သည် ...",
#     tools=[receipted_lookup],   # သိမ်းဆည်းထားသော ကိရိယာ၊ မူရင်းလုပ်ဆောင်ချက်မဟုတ်ပါ
# )
# response = agent.run("ဇွန်လတွင် Sydney မှ Los Angeles သို့ ဇယားရှာပါ။")
#
# # လုပ်ဆောင်ပြီးနောက် အေးဂျင့်ဖြင့်ခေါ်ယူသော ကိရိယာတိုင်းတွင် လက်မှတ်ထိုးထားသော ကဒ်ထပ်ရှိသည်။
# audit_chain = receipted_lookup.receipts
```

Agent framework သည် receipt များအကြောင်း ဘာမှ မသိရန်အတွက် လိုအပ်မှု မရှိပါ။ Receipt အတည်ပြုခြင်းသည် tool အကျော့ ဝိုင်းထားပြီး framework တွင် ထည့်သွင်းထားခြင်း မဟုတ်ပါ။ ၎င်းဖြင့် agent ကို ပြန်ရေးသားခြင်းမပြုဘဲ ရှိပြီးသား agent code တွင် provenance ဖြည့်စွက်နိုင်ပါသည်။


## ပြန်လည်သုံးသပ်ခြင်းနှင့် စိန်ခေါ်မှုတိုးခြင်း

သင်မှာရှိသည်မှာ -

- Ed25519 key pair တစ်စုံဖန်တီးခဲ့သည်။
- agent tool ခေါ်ဆိုမှုအတွက် receipt တစ်စီပြုလုပ်ပြီး လက်မှတ်ရေးထိုးခဲ့သည်။
- ပြည်သူ့ key ဖြင့်သာ offline တွင် receipt ကိုစစ်ဆေးနိုင်ခဲ့သည်။
- receipt တစ်စောင်ကို ပုံမှန်မဟုတ်စွာပြင်ဆင်ပြီး စစ်ဆေးမှု မအောင်မြင်ကြောင်းတွေ့ရှိခဲ့သည်။
- receipt သုံးခုကို hash-chained စဉ်ဆက်အဖြစ်တည်ဆောက်ခဲ့သည်။
- လမ်းကြောင်းအလယ်တန်းကို လိမ်လည်ပြင်ဆင်ပြီး လက်မှတ်ရိုက်မှတ်မှုနှင့် လမ်းကြောင်းထိန်းချုပ်မှု နှစ်ခုစလုံး မအောင်မြင်ကြောင်းတွေ့ရှိခဲ့သည်။
- tool function ကို အလိုအလျောက် receipt လက်မှတ်ရေးထိုးမှုဖြင့် ဝတ်စားခတ်ထားသည်။

**စိန်ခေါ်မှု တိုးချဲ့ခြင်း။** receipt schema ကို `request_id` ရာထူးဖြင့် တိုးမြှင့်ပါ (distribute လုပ်သတ်မှတ်မှုအတွက် UUID ဖြစ်သည်)။ `make_receipt` သို့ ထည့်သွင်းပြီး receipt များကို အဆုံးအထိ စစ်ဆေးနိုင်ကြောင်းအတည်ပြုပါ။ ထို့နောက် လက်မှတ်ရေးထိုးပြီးနောက် field ကိုပြင်ပြီး စစ်ဆေးမှု မအောင်မြင်ကြောင်း အတည်ပြုပါ။ ၎င်းက canonical encoding ၏ byte များစွာသည် လက်မှတ်ရေးထိုးမှုတွင် မည်သို့ပါဝင်သည်ကို နားလည်ဖို့အတွက် ကြိုးစားခိုင်းသည်။

**အရေးကြီးသော အကန့်အသတ်။** Receipt များသည် အချက်သုံးခုသာ ပြသပေးသည် - attribution (ဤ key သည် ထို content ကို လက်မှတ်ရေးထိုးခဲ့သည်), integrity (content သည် လက်မှတ်ရေးထိုးပြီးနောက် မပြောင်းလဲထားပါ), နှင့် ordering (ဤ receipt သည် ထို receipt ထက် နောက်ပိုင်း ရောက်ရှိခြင်း) ဖြစ်သည်။ ၎င်းသည် agent ၏လုပ်ဆောင်မှုမှန်ကန်မှုကို မပြသပါ၊ `policy_id` တွင် ဖော်ပြထားသော policy ကို အမှန်တကယ် သုံးသပ်ထားကြောင်း မထောက်ခံပါ၊ agent သည် စည်းကမ်းခွဲများကို လိုက်နာခြင်း မပြုထားကြောင်းလည်း မထောက်ခံပါ။ Receipt များသည် အခြေခံဖြစ်သည်။ အုပ်ချုပ်ရေးသည် သင်တည်ဆောက်လိုက်သည့် စနစ်ဖြစ်သည်။

ဒီအကန့်အသတ်ကို ထည့်သတိပြု၍ သင်ခန်းစာ README ကို ပြန်လည်ဖတ်ရှုပါ။ Receipt များအတွက် အများဆုံးဖြစ်ပျက်ရသောအမှားမှာ "we have receipts" ဟု ထင်ခြင်းအားဖြင့် "we are governed" ဖြစ်သည်ဟု မှတ်ယူခြင်းဖြစ်သည်။ ၎င်းမဟုတ်ပါ။ Receipt များသည် agent ၏သွင်ပြင်အပြုအမူကို စစ်ဆေးနိုင်အောင် လုပ်ပေးသည်။ ၎င်းများသည် မှန်ကန်မှုကို မပြုလုပ်ပေးပါ။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
